# ROCKET и MiniRocket на крупных UCR-датасетах

**ТОЛЬКО** ROCKET и MiniRocket (без BOSS, который очень медленный на крупных датасетах).
Датасеты: FordA, Wafer, ElectricDevices.

Runtime: ≈ 5–10 мин.
Output: `ucr_rocket_large_results.json` + `.csv`

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','--quiet','aeon','sktime'])
print('OK')

In [ ]:
import json, time, warnings
import numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
warnings.filterwarnings('ignore')

from aeon.datasets import load_classification
from aeon.classification.convolution_based import RocketClassifier, MiniRocketClassifier

DATASETS = ['FordA', 'Wafer', 'ElectricDevices']
SEED = 42
results = []

def evaluate(name, model, Xtr, ytr, Xte, yte, ds):
    t0 = time.time()
    model.fit(Xtr, ytr)
    train_sec = time.time() - t0
    t0 = time.time()
    pred = model.predict(Xte)
    inf_sec = time.time() - t0
    acc = float(accuracy_score(yte, pred))
    bal = float(balanced_accuracy_score(yte, pred))
    f1m = float(f1_score(yte, pred, average='macro'))
    print(f'  {name:>14} acc={acc:.4f} bal={bal:.4f} f1m={f1m:.4f} train={train_sec:.1f}s inf={inf_sec:.1f}s')
    return {'dataset':ds,'method':name,'accuracy':acc,'balanced_accuracy':bal,'macro_f1':f1m,'train_seconds':train_sec,'infer_seconds':inf_sec}

def majority(ytr, yte):
    cls,cnt=np.unique(ytr,return_counts=True)
    pred=np.full(len(yte), cls[np.argmax(cnt)])
    return {'accuracy':float(accuracy_score(yte,pred)),'balanced_accuracy':float(balanced_accuracy_score(yte,pred)),'macro_f1':float(f1_score(yte,pred,average='macro'))}

In [ ]:
for ds in DATASETS:
    print(f'\n=== {ds} ===')
    Xtr,ytr=load_classification(ds, split='train')
    Xte,yte=load_classification(ds, split='test')
    n_train=Xtr.shape[0]; L=Xtr.shape[-1]; n_classes=len(np.unique(ytr))
    print(f'  train={n_train} test={Xte.shape[0]} len={L} classes={n_classes}')
    
    m = majority(ytr, yte)
    print(f'  {"Majority":>14} acc={m["accuracy"]:.4f} bal={m["balanced_accuracy"]:.4f} f1m={m["macro_f1"]:.4f}')
    results.append({'dataset':ds,'method':'Majority',**m,'train_seconds':0.0,'infer_seconds':0.0})
    
    try: results.append(evaluate('ROCKET', RocketClassifier(random_state=SEED), Xtr,ytr,Xte,yte,ds))
    except Exception as e: print(f'  ROCKET FAIL: {e}')
    
    try: results.append(evaluate('MiniRocket', MiniRocketClassifier(random_state=SEED), Xtr,ytr,Xte,yte,ds))
    except Exception as e: print(f'  MiniRocket FAIL: {e}')
    
    with open('ucr_rocket_large_results.json','w') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    pd.DataFrame(results).to_csv('ucr_rocket_large_results.csv', index=False)

print('\nDONE')

In [ ]:
df = pd.DataFrame(results)
piv = df.pivot_table(index='dataset', columns='method', values='accuracy')
print(piv.round(4))